# Intermediate 01 — Workload Identity with SPIFFE & SPIRE

## Scenario

Our enterprise has:

- `agent:travel-booking` — governed logical agent;
- a production Kubernetes workload running that agent;
- `payment-tool` — a sensitive downstream service.

We want to replace static service secrets with **attested, short-lived workload identity**.

This notebook combines runnable Python with real SPIRE deployment exercises.


In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from urllib.parse import urlparse
import json, os, ssl, socket, tempfile, threading, time

def now():
    return datetime.now(timezone.utc)


## 1 — Parse SPIFFE IDs

In [ ]:
def parse_spiffe_id(value: str):
    u = urlparse(value)
    if u.scheme != "spiffe":
        raise ValueError("SPIFFE ID must use spiffe://")
    if not u.netloc:
        raise ValueError("missing trust domain")
    if u.query or u.fragment:
        raise ValueError("query/fragment not allowed in this lab")
    return {"trust_domain": u.netloc, "path": u.path or "/"}

ids = [
    "spiffe://corp.example/prod/agent/travel-booking",
    "spiffe://corp.example/prod/tool/payment",
]
for x in ids:
    print(x, "=>", parse_spiffe_id(x))


## 2 — Map logical agent to approved workload

In [ ]:
AGENT_REGISTRY = {
    "agent:travel-booking": {
        "approved_workload_ids": {
            "spiffe://corp.example/prod/agent/travel-booking"
        },
        "risk": "high",
    }
}

def workload_matches_agent(logical_agent, workload_spiffe_id):
    return workload_spiffe_id in AGENT_REGISTRY[logical_agent]["approved_workload_ids"]

print(workload_matches_agent(
    "agent:travel-booking",
    "spiffe://corp.example/prod/agent/travel-booking"
))
print(workload_matches_agent(
    "agent:travel-booking",
    "spiffe://corp.example/dev/agent/travel-booking"
))


## 3 — Selector-based workload attestation model

In [ ]:
REGISTRATION = [
    {
        "spiffe_id": "spiffe://corp.example/prod/agent/travel-booking",
        "selectors": {
            "k8s:ns": "agents",
            "k8s:sa": "travel-booking",
        },
    },
    {
        "spiffe_id": "spiffe://corp.example/prod/tool/payment",
        "selectors": {
            "k8s:ns": "tools",
            "k8s:sa": "payment-tool",
        },
    },
]

def attest_workload(observed_selectors):
    matches = []
    for entry in REGISTRATION:
        if all(observed_selectors.get(k) == v for k, v in entry["selectors"].items()):
            matches.append(entry["spiffe_id"])
    return matches

print(attest_workload({"k8s:ns":"agents", "k8s:sa":"travel-booking"}))
print(attest_workload({"k8s:ns":"default", "k8s:sa":"travel-booking"}))


The caller does not choose its identity. Trusted runtime evidence is matched against registration policy.

## 4 — Demonstrate why broad selectors are dangerous

In [ ]:
broad_entry = {
    "spiffe_id": "spiffe://corp.example/prod/agent/admin",
    "selectors": {"k8s:ns": "agents"},
}

def matches(entry, observed):
    return all(observed.get(k) == v for k, v in entry["selectors"].items())

attacker = {"k8s:ns": "agents", "k8s:sa": "unrelated-pod"}
print("Attacker matches broad identity rule:", matches(broad_entry, attacker))


## 5 — Create short-lived X.509 lab identities

In [ ]:
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa

def make_ca():
    key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    name = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, "Lab SPIFFE CA")])
    cert = (
        x509.CertificateBuilder()
        .subject_name(name).issuer_name(name)
        .public_key(key.public_key())
        .serial_number(x509.random_serial_number())
        .not_valid_before(now() - timedelta(minutes=1))
        .not_valid_after(now() + timedelta(days=1))
        .add_extension(x509.BasicConstraints(ca=True, path_length=None), critical=True)
        .sign(key, hashes.SHA256())
    )
    return key, cert

def issue_workload_cert(ca_key, ca_cert, spiffe_id, ttl_minutes=10):
    key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    subject = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, "workload")])
    cert = (
        x509.CertificateBuilder()
        .subject_name(subject).issuer_name(ca_cert.subject)
        .public_key(key.public_key())
        .serial_number(x509.random_serial_number())
        .not_valid_before(now() - timedelta(seconds=10))
        .not_valid_after(now() + timedelta(minutes=ttl_minutes))
        .add_extension(
            x509.SubjectAlternativeName([
                x509.UniformResourceIdentifier(spiffe_id)
            ]),
            critical=False,
        )
        .sign(ca_key, hashes.SHA256())
    )
    return key, cert

ca_key, ca_cert = make_ca()
agent_key, agent_cert = issue_workload_cert(
    ca_key, ca_cert,
    "spiffe://corp.example/prod/agent/travel-booking"
)
tool_key, tool_cert = issue_workload_cert(
    ca_key, ca_cert,
    "spiffe://corp.example/prod/tool/payment"
)

print(agent_cert.not_valid_after_utc)


## 6 — Extract the SPIFFE URI SAN

In [ ]:
def extract_spiffe_id(cert):
    san = cert.extensions.get_extension_for_class(
        x509.SubjectAlternativeName
    ).value
    uris = san.get_values_for_type(x509.UniformResourceIdentifier)
    spiffe = [u for u in uris if u.startswith("spiffe://")]
    if len(spiffe) != 1:
        raise ValueError("expected exactly one SPIFFE URI SAN")
    return spiffe[0]

print(extract_spiffe_id(agent_cert))
print(extract_spiffe_id(tool_cert))


## 7 — Authorization consumes authenticated workload identity

In [ ]:
AUTHZ = {
    "spiffe://corp.example/prod/agent/travel-booking": {
        ("payment:create", "travel:approved")
    }
}

def authorize(principal, action, resource_class):
    return (action, resource_class) in AUTHZ.get(principal, set())

principal = extract_spiffe_id(agent_cert)
print(authorize(principal, "payment:create", "travel:approved"))
print(authorize(principal, "iam:grant_admin", "production"))


SPIFFE authenticated the workload. Our policy still decides what that workload may do.

## 8 — Rotation without changing identity

In [ ]:
old_serial = agent_cert.serial_number
agent_key_2, agent_cert_2 = issue_workload_cert(
    ca_key, ca_cert,
    "spiffe://corp.example/prod/agent/travel-booking",
    ttl_minutes=10,
)

print("same SPIFFE ID:", extract_spiffe_id(agent_cert_2) == extract_spiffe_id(agent_cert))
print("different certificate:", old_serial != agent_cert_2.serial_number)


## 9 — Model a Workload API update stream

In [ ]:
@dataclass
class SVIDUpdate:
    spiffe_id: str
    generation: int
    expires_at: datetime

def simulated_workload_api():
    for generation in range(1, 4):
        yield SVIDUpdate(
            "spiffe://corp.example/prod/agent/travel-booking",
            generation,
            now() + timedelta(minutes=10),
        )

for update in simulated_workload_api():
    print(update)


Production applications should be prepared for identity/trust updates instead of loading a certificate once at startup.

## 10 — JWT-SVID claim model

In [ ]:
import jwt

JWT_KEY = "lab-only-signing-secret"

def issue_lab_jwt_svid(spiffe_id, audience, ttl_seconds=120):
    t = now()
    return jwt.encode({
        "sub": spiffe_id,
        "aud": audience,
        "iat": int(t.timestamp()),
        "exp": int((t + timedelta(seconds=ttl_seconds)).timestamp()),
    }, JWT_KEY, algorithm="HS256")

token = issue_lab_jwt_svid(
    "spiffe://corp.example/prod/agent/travel-booking",
    "payment-api",
)
claims = jwt.decode(
    token, JWT_KEY,
    algorithms=["HS256"],
    audience="payment-api",
)
print(claims)


## 11 — Audience prevents token reuse at another service

In [ ]:
try:
    jwt.decode(
        token, JWT_KEY,
        algorithms=["HS256"],
        audience="admin-api",
    )
except Exception as e:
    print("DENIED:", type(e).__name__, str(e))


This is a claim-model demonstration, not a replacement for a real SPIRE JWT-SVID implementation.

## 12 — Trust-domain authorization

In [ ]:
def trust_domain_of(spiffe_id):
    return parse_spiffe_id(spiffe_id)["trust_domain"]

def accepted_partner(principal, accepted_domains):
    return trust_domain_of(principal) in accepted_domains

print(accepted_partner(
    "spiffe://partner.example/agent/research",
    {"corp.example", "partner.example"},
))


## 13 — Federation is not blanket authorization

In [ ]:
FEDERATED_DOMAINS = {"partner.example"}

PARTNER_AUTHZ = {
    "spiffe://partner.example/agent/research": {"docs:public:read"}
}

principal = "spiffe://partner.example/agent/research"

authenticated = trust_domain_of(principal) in FEDERATED_DOMAINS
authorized_public = "docs:public:read" in PARTNER_AUTHZ.get(principal, set())
authorized_payments = "payments:create" in PARTNER_AUTHZ.get(principal, set())

print("federated identity accepted:", authenticated)
print("public docs:", authorized_public)
print("payments:", authorized_payments)


## 14 — Kubernetes registration-entry design

In [ ]:
k8s_entries = [
    {
        "spiffe_id": "spiffe://corp.example/prod/agent/travel-booking",
        "selectors": [
            "k8s:ns:agents",
            "k8s:sa:travel-booking",
        ],
    },
    {
        "spiffe_id": "spiffe://corp.example/prod/tool/payment",
        "selectors": [
            "k8s:ns:tools",
            "k8s:sa:payment-tool",
        ],
    },
]

print(json.dumps(k8s_entries, indent=2))


## 15 — Real SPIRE lab: architecture

On a Docker/Kubernetes-capable machine, install SPIRE and create:

```text
SPIRE Server
SPIRE Agent
travel workload
payment workload
```

Useful official commands to explore include:

```bash
spire-server run -config server.conf
spire-agent run -config agent.conf
spire-server entry create ...
spire-agent api fetch x509 -socketPath ...
spire-agent api fetch jwt -audience payment-api -socketPath ...
```

Exact configuration varies by SPIRE version and attestation plugin. Follow the current SPIRE docs rather than copying production configuration blindly from this notebook.

### Verification goals

After setup, verify that:

```text
travel workload -> travel SPIFFE ID
payment workload -> payment SPIFFE ID
unregistered workload -> receives neither identity
```


## 16 — Inspect a real X.509-SVID

In [ ]:
# After exporting a real SVID to a PEM file:
#
# from cryptography import x509
# cert = x509.load_pem_x509_certificate(open("svid.pem","rb").read())
# print(cert.subject)
# print(cert.issuer)
# print(extract_spiffe_id(cert))
# print(cert.not_valid_after_utc)
#
print("Exercise cell: provide svid.pem from your SPIRE environment.")


## 17 — Real mTLS experiment

Build two small services:

```text
agent-client
payment-server
```

Requirements:

1. each retrieves its X.509-SVID from the Workload API;
2. both trust the SPIFFE bundle;
3. TLS requires client and server certificates;
4. server extracts caller SPIFFE ID;
5. server authorizes only:

```text
spiffe://corp.example/prod/agent/travel-booking
```

6. rotate the client SVID while the application remains running.

Observe:

```text
logical identity stays stable
certificate serial changes
mTLS continues
```


## 18 — Adversarial selector tests

In [ ]:
tests = [
    (
        {"k8s:ns":"agents", "k8s:sa":"travel-booking"},
        ["spiffe://corp.example/prod/agent/travel-booking"],
    ),
    (
        {"k8s:ns":"agents", "k8s:sa":"evil"},
        [],
    ),
    (
        {"k8s:ns":"tools", "k8s:sa":"travel-booking"},
        [],
    ),
]

for observed, expected in tests:
    actual = attest_workload(observed)
    assert actual == expected, (observed, actual, expected)

print("Attestation regression tests passed.")


## 19 — Exercise: identity naming

Design SPIFFE IDs for:

```text
production research agent
production RAG retriever
payment MCP server
policy decision service
staging research agent
```

Then answer:

- Should staging and production share a trust domain?
- Which parts of the path should be stable?
- Should individual pods receive different logical IDs?
- What does authorization gain from your naming scheme?

## 20 — Exercise: agent registry integration

Extend:

```python
AGENT_REGISTRY
```

with:

```text
owner
sponsor
risk
approved workload identities
allowed trust domains
allowed tools
```

Implement:

```python
verify_runtime_actor(logical_agent, spiffe_id)
```

A request should be accepted only when:

```text
logical agent is active
AND
workload SPIFFE ID is approved
AND
workload trust domain is approved
```

## 21 — Exercise: compromised neighboring workload

Threat model:

```text
attacker compromises another pod in namespace agents
```

Try these selector strategies:

```text
namespace only
service account only
namespace + service account
namespace + service account + additional attested attribute
```

Compare the impersonation surface.

## 22 — Exercise: federation

Design federation between:

```text
spiffe://enterprise.example
spiffe://vendor.example
```

The vendor research agent may call only:

```text
enterprise public-search tool
```

Explain separately:

```text
bundle exchange
authentication
authorization
revocation
audit
```

## 23 — Exercise: credential broker

Design:

```text
SPIFFE X.509-SVID
      |
      v
Credential Broker
      |
      v
short-lived SaaS OAuth token
```

The LLM must never receive:

```text
SPIFFE private key
broker credential
refresh token
```

Which component performs each exchange?

## Review questions

1. What is the difference between SPIFFE and SPIRE?
2. Is a SPIFFE ID a credential?
3. What is an SVID?
4. Why are X.509-SVIDs useful for mTLS?
5. Why do JWT-SVIDs require careful audience design?
6. What problem does the Workload API solve?
7. How can a workload obtain identity without a bootstrap application secret?
8. What is node attestation?
9. What is workload attestation?
10. Why can broad selectors destroy identity assurance?
11. Why must applications handle SVID rotation?
12. What is a trust bundle?
13. What does federation establish?
14. Why does federation not imply authorization?
15. How should logical agent identity relate to SPIFFE workload identity?
16. How can SPIFFE reduce static downstream credentials?

# Next course

## Intermediate 02 — OAuth 2.x and OpenID Connect for Agents
